# Step 2: measure official ByteTrack on MOT17 half-validation

Start with **MOT17-09-FRCNN**, using original JPEG frames and the authors' YOLOX-X ablation checkpoint. This is a one-sequence pipeline check, not the full paper reproduction. Once it works, you can select all seven unique training scenes.

Upload this notebook into Colab and select a GPU. It is standalone: run from the top even if your earlier runtime disconnected. The official image archive is approximately **5.5 GB** and downloads directly into Colab. Allow time and runtime storage for downloading and extracting selected validation images. This notebook does not mount Drive. Download its result ZIP before disconnecting.

The authors' model, preprocessing and BYTETracker are reused. A small inference/evaluation wrapper handles original JPEG frames, explicit one-based frame numbers, empty detection frames, and TrackEval. It records these choices instead of claiming to execute the original track.py evaluator unchanged.

Sources: [ByteTrack](https://arxiv.org/abs/2110.06864), [official code](https://github.com/FoundationVision/ByteTrack), [MOT17 data](https://motchallenge.net/data/MOT17/), [TrackEval](https://github.com/JonathonLuiten/TrackEval).

This notebook is an AI-assisted learning tool. Follow your course's submission rules. Static checks were performed locally; the new evaluation notebook has not yet been executed on Colab GPU.


## 1. Check the GPU
If this fails, select a GPU runtime and rerun. Keep Colab's matching PyTorch and torchvision packages.


In [ ]:
import sys, subprocess, json, hashlib, re, os, time, shutil
from pathlib import Path
import torch, torchvision
assert torch.cuda.is_available(), 'Choose a GPU under Runtime > Change runtime type.'
print('Python:', sys.version)
print('PyTorch:', torch.__version__, 'torchvision:', torchvision.__version__)
print('GPU:', torch.cuda.get_device_name(0))


## 2. Install demo dependencies
The old repository requirements include dated optional ONNX packages. This installs the dependencies for the PyTorch video demo instead. It does not change the tracking algorithm or replace PyTorch. If Colab requests a restart, restart and rerun from Step 1.


In [ ]:
packages = ['loguru', 'scikit-image', 'tqdm', 'Pillow', 'thop', 'ninja', 'tabulate',
            'tensorboard', 'lap', 'motmetrics', 'filterpy', 'h5py', 'pycocotools',
            'cython_bbox', 'gdown', 'pandas', 'matplotlib', 'scipy']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)


## 3. Get the official source and record compatibility edits
The source revision is fixed. Running from the source directory avoids building the optional accelerated COCO evaluator, which this demo does not use. Deprecated `np.float` aliases become `float`, the GUI key wait is disabled, and checkpoint loading explicitly uses PyTorch's restricted weights loader. All edits are saved in a patch file.


In [ ]:
REVISION = 'd1bf0191adff59bc8fcfeaa0b33d3d1642552a99'
REPO = Path('/content/ByteTrack_paper_starter')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/FoundationVision/ByteTrack.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', REVISION], check=True)
for path in (REPO / 'yolox/tracker').glob('*.py'):
    original = path.read_text()
    patched = re.sub(r'np\.float\b', 'float', original)
    if patched != original:
        path.write_text(patched)
demo = REPO / 'tools/demo_track.py'
source = demo.read_text()
source = source.replace('ch = cv2.waitKey(1)', 'ch = -1  # Colab has no GUI keyboard window')
source = source.replace('torch.load(ckpt_file, map_location="cpu")',
                        'torch.load(ckpt_file, map_location="cpu", weights_only=True)')
demo.write_text(source)
OUT = Path('/content/bytetrack_paper_first_run')
OUT.mkdir(exist_ok=True)
patch = subprocess.check_output(['git', '-C', str(REPO), 'diff'], text=True)
(OUT / 'compatibility.patch').write_text(patch)
print(patch)
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO) + os.pathsep + env.get('PYTHONPATH', '')
subprocess.run([sys.executable, '-c',
    'from yolox.exp import get_exp; from yolox.tracker.byte_tracker import BYTETracker; print("Official imports OK")'],
    cwd=REPO, env=env, check=True)


## 4. Download the official ablation checkpoint
The file ID below comes from the official README's ablation model link. It is paired with `yolox_x_ablation.py`. The checkpoint is several hundred MB. If Google Drive rejects the automated download, use the README link to download it manually and upload it into the printed destination using Colab's Files panel, then rerun this cell.

Use only the official checkpoint. A restricted checkpoint-loading error should be inspected rather than bypassed with an unknown file. This model's documented protocol is MOT17 half-training plus CrowdHuman, followed by MOT17 half-validation. This notebook does not download or evaluate that dataset yet.


In [ ]:
import gdown
WEIGHTS = REPO / 'pretrained/bytetrack_ablation.pth.tar'
WEIGHTS.parent.mkdir(exist_ok=True)
if not WEIGHTS.exists():
    downloaded = gdown.download(id='1iqhM-6V_r1FpOlOzrdP_Ejshgk0DxOob', output=str(WEIGHTS), quiet=False)
    if downloaded is None:
        raise RuntimeError(f'Download failed. Manually upload the official checkpoint to {WEIGHTS}.')
assert WEIGHTS.is_file() and WEIGHTS.stat().st_size > 10_000_000, f'Incomplete checkpoint: {WEIGHTS}'
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
weight_hash = sha256_file(WEIGHTS)
print('Checkpoint:', WEIGHTS, 'bytes:', WEIGHTS.stat().st_size)
print('SHA256:', weight_hash, '(recorded for repeatability, not a publisher-verified checksum)')


## 5. Choose the evaluation scenes and download MOT17
Start with ALL_SEQUENCES=False. Do not tune settings after seeing evaluation results and then describe those results as an untouched test.

The original repository's converter selects only FRCNN-named copies for private-detector evaluation. These are scene names: this notebook still runs its own YOLOX detector, not the provided FRCNN detections. DPM/FRCNN/SDP variants must not be counted as three independent videos.

If the automatic download fails, use the official MOT17 page to download MOT17.zip, then place the file at /content/MOT17.zip and rerun. The archive URL is the official page's download link.


In [ ]:
import urllib.request, zipfile, configparser, csv
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
ALL_SEQUENCES = False #@param {type:"boolean"}
scene_numbers = ['02','04','05','09','10','11','13'] if ALL_SEQUENCES else ['09']
SEQUENCES = [f'MOT17-{x}-FRCNN' for x in scene_numbers]
WORK = Path('/content/bytetrack_mot17_halfval')
WORK.mkdir(exist_ok=True)
shutil.copyfile(OUT / 'compatibility.patch', WORK / 'bytetrack_compatibility.patch')
ARCHIVE = Path('/content/MOT17.zip')
if not ARCHIVE.is_file():
    partial = ARCHIVE.with_suffix('.zip.part')
    last_percent = [-10]
    def progress(block, size, total):
        if total > 0:
            percent = min(100, int(block * size * 100 / total))
            if percent >= last_percent[0] + 10:
                print(f'MOT17 download: {percent}%')
                last_percent[0] = percent
    urllib.request.urlretrieve('https://motchallenge.net/data/MOT17.zip', str(partial), reporthook=progress)
    assert zipfile.is_zipfile(partial), 'Download was not a ZIP. Check the official download page.'
    partial.rename(ARCHIVE)
assert zipfile.is_zipfile(ARCHIVE), 'Invalid archive; replace it with the official MOT17.zip.'
print('Selected:', SEQUENCES)
print('Archive size, GB:', round(ARCHIVE.stat().st_size / 1e9, 2))


## 6. Extract the exact validation range and preserve ground truth
The official converter uses zero-based validation indices N//2+1 through N-1. Therefore the first original one-based frame is **N//2+2**. For MOT17-09 (525 frames), this means original frames **264–525**, or **262 validation frames**.

We reset tracker state at validation start. Ground truth and predictions both use new one-based validation frame indices. All ground-truth classes and ignore flags remain in the files for standard MOT17 preprocessing. No video compression or coordinate scaling occurs.


In [ ]:
info_by_seq = {}
with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    for seq in SEQUENCES:
        candidates = [n for n in names if n.endswith(f'train/{seq}/seqinfo.ini')]
        assert len(candidates) == 1, f'Cannot uniquely find {seq}/seqinfo.ini in the official archive.'
        ini_name = candidates[0]
        prefix = ini_name[:-len('seqinfo.ini')]
        cfg = configparser.ConfigParser()
        cfg.read_string(archive.read(ini_name).decode('utf-8-sig'))
        s = cfg['Sequence']
        n, w, h, fps = int(s['seqLength']), int(s['imWidth']), int(s['imHeight']), int(s['frameRate'])
        first_frame = n // 2 + 2
        count = n - first_frame + 1
        folder = WORK / 'images' / seq
        folder.mkdir(parents=True, exist_ok=True)
        for original_frame in range(first_frame, n + 1):
            member = prefix + f'img1/{original_frame:06d}.jpg'
            target = folder / f'{original_frame:06d}.jpg'
            if not target.is_file():
                target.write_bytes(archive.read(member))
        full_gt = np.loadtxt(archive.open(prefix + 'gt/gt.txt'), delimiter=',', ndmin=2)
        assert full_gt.shape[1] >= 9
        gt = full_gt[(full_gt[:,0] >= first_frame) & (full_gt[:,0] <= n)].copy()
        gt[:,0] -= first_frame - 1
        assert len(gt) and gt[:,0].min() >= 1 and gt[:,0].max() <= count
        gt_folder = WORK / 'gt' / seq
        (gt_folder / 'gt').mkdir(parents=True, exist_ok=True)
        np.savetxt(gt_folder / 'gt/gt.txt', gt, delimiter=',', fmt='%.8f')
        cfg['Sequence']['seqLength'] = str(count)
        with (gt_folder / 'seqinfo.ini').open('w') as f: cfg.write(f)
        info_by_seq[seq] = dict(original_frames=n, first_original_frame=first_frame,
                                validation_frames=count, width=w, height=h, fps=fps,
                                images=str(folder), gt_rows=len(gt))
(WORK / 'split_info.json').write_text(json.dumps(info_by_seq, indent=2))
display(pd.DataFrame(info_by_seq).T)


## 7. Inspect annotation alignment before scoring
The green boxes below are scored pedestrian ground-truth boxes. Check that they enclose the correct people on the selected first, middle and last validation frames. If boxes or frames do not match, stop and inspect the input. The full ignore/distractor annotations are retained for evaluation, although they are not drawn here.


In [ ]:
fig, axes = plt.subplots(len(SEQUENCES), 3, figsize=(18, 4 * len(SEQUENCES)), squeeze=False)
for row, seq in enumerate(SEQUENCES):
    info = info_by_seq[seq]
    gt = np.loadtxt(WORK / 'gt' / seq / 'gt/gt.txt', delimiter=',', ndmin=2)
    for col, frame_id in enumerate([1, (info['validation_frames']+1)//2, info['validation_frames']]):
        original = info['first_original_frame'] + frame_id - 1
        image = cv2.imread(str(Path(info['images']) / f'{original:06d}.jpg'))
        assert image is not None and image.shape[:2] == (info['height'], info['width'])
        for r in gt[(gt[:,0]==frame_id) & (gt[:,6]==1) & (gt[:,7]==1)]:
            x,y,w,h = r[2:6]
            cv2.rectangle(image,(round(x),round(y)),(round(x+w),round(y+h)),(0,255,0),2)
        axes[row,col].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
        axes[row,col].set_title(f'{seq}: original {original}, validation {frame_id}')
        axes[row,col].axis('off')
plt.tight_layout()
plt.show()


In [ ]:
ALIGNMENT_CHECKED = False #@param {type:"boolean"}
assert ALIGNMENT_CHECKED, 'Inspect the green annotation boxes above, then check ALIGNMENT_CHECKED and rerun.'


## 8. Run official YOLOX and BYTETracker on validation images
The wrapper uses the official Predictor preprocessing and postprocessing and official BYTETracker. Tracker settings follow track.py defaults: track_thresh=0.6, match_thresh=0.9, track_buffer=30 (14 for MOT17-05, 25 for MOT17-13). It applies the original evaluator's minimum area and aspect ratio output filters.

It advances the tracker on every frame, including an empty detection array when no boxes exist. This fixes a timing issue in the old demo rather than skipping empty frames. The original class performs both association stages unchanged. The original script's COCO AP calculation is omitted; TrackEval evaluates the exported MOT tracks.

The original evaluator has additional implementation and environment details. These results are a documented reproduction attempt, not a guarantee of the paper's exact numbers. Start with one scene before spending time on the full set.


In [ ]:
sys.path.insert(0, str(REPO))
from types import SimpleNamespace
from yolox.exp import get_exp
from yolox.utils import fuse_model
from yolox.tracker.byte_tracker import BYTETracker
from yolox.tracking_utils.timer import Timer
from tools.demo_track import Predictor

exp = get_exp(str(REPO / 'exps/example/mot/yolox_x_ablation.py'), None)
exp.test_conf = 0.01  # track.py default; second association itself keeps scores > 0.1
exp.nmsthre = 0.7
model = exp.get_model()
checkpoint = torch.load(WEIGHTS, map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint['model'])
del checkpoint
model = fuse_model(model).eval().cuda().half()
predictor = Predictor(model, exp, device=torch.device('cuda'), fp16=True)
# Warm up outside the recorded loop.
with torch.no_grad():
    model(torch.zeros((1,3,*exp.test_size),device='cuda',dtype=torch.float16))
torch.cuda.synchronize()
run_info = {}
for seq in SEQUENCES:
    info = info_by_seq[seq]
    buffer = 14 if seq == 'MOT17-05-FRCNN' else (25 if seq == 'MOT17-13-FRCNN' else 30)
    args = SimpleNamespace(track_thresh=0.6, track_buffer=buffer, match_thresh=0.9, mot20=False)
    # The original evaluator initializes BYTETracker at its default frame_rate=30.
    tracker = BYTETracker(args)
    timer = Timer()
    output = WORK / 'trackers' / 'official_bytetrack' / 'data' / f'{seq}.txt'
    output.parent.mkdir(parents=True, exist_ok=True)
    detector_seconds = 0.0
    tracker_seconds = 0.0
    written = 0
    processed = 0
    start_all = time.perf_counter()
    with output.open('w', newline='') as handle:
        writer = csv.writer(handle)
        for frame_id in range(1, info['validation_frames'] + 1):
            original = info['first_original_frame'] + frame_id - 1
            image = cv2.imread(str(Path(info['images']) / f'{original:06d}.jpg'))
            assert image is not None and image.shape[:2] == (info['height'], info['width'])
            torch.cuda.synchronize()
            start = time.perf_counter()
            outputs, img_info = predictor.inference(image, timer)
            torch.cuda.synchronize()
            detector_seconds += time.perf_counter() - start
            det = outputs[0] if outputs[0] is not None else np.empty((0,5),dtype=np.float32)
            start = time.perf_counter()
            tracks = tracker.update(det, [info['height'], info['width']], exp.test_size)
            torch.cuda.synchronize()
            tracker_seconds += time.perf_counter() - start
            for track in tracks:
                x,y,w,h = track.tlwh
                if w*h > 100 and h > 0 and w/h <= 1.6:
                    writer.writerow([frame_id, track.track_id, f'{x:.2f}', f'{y:.2f}',
                                     f'{w:.2f}', f'{h:.2f}', f'{track.score:.2f}', -1,-1,-1])
                    written += 1
            timer.toc()
            processed += 1
            if frame_id % 100 == 0: print(seq, frame_id, '/', info['validation_frames'])
    assert processed == info['validation_frames']
    run_info[seq] = dict(frames=processed, boxes_written=written,
                         detector_seconds=detector_seconds, tracker_seconds=tracker_seconds,
                         pipeline_wall_seconds=time.perf_counter()-start_all,
                         tracker_args=vars(args), test_size=list(exp.test_size),
                         detector_confidence=exp.test_conf, nms=exp.nmsthre)
    print(seq, 'completed:', processed, 'frames;', written, 'tracked boxes')
(WORK / 'inference_info.json').write_text(json.dumps(run_info, indent=2))


## 9. Install the pinned TrackEval evaluator and compute metrics
Standard MOT17 preprocessing is enabled to handle ignore classes. A small NumPy alias migration is recorded for this older evaluator revision. MOTA, IDF1 and HOTA are shown as percentages. Higher is better. ID switches, false positives and false negatives are counts; lower is better. The one-sequence score should not be compared directly with the paper's aggregate score.


In [ ]:
EVAL_REVISION = '12c8791b303e0a0b50f753af204249e622d0281a'
EVAL_REPO = Path('/content/TrackEval_paper_step2')
if not (EVAL_REPO / '.git').exists():
    subprocess.run(['git','clone','https://github.com/JonathonLuiten/TrackEval.git',str(EVAL_REPO)],check=True)
subprocess.run(['git','-C',str(EVAL_REPO),'checkout',EVAL_REVISION],check=True)
for path in (EVAL_REPO / 'trackeval').rglob('*.py'):
    source = path.read_text()
    patched = re.sub(r'np\.(float|int|bool)\b', lambda m: m.group(1), source)
    if source != patched: path.write_text(patched)
(WORK / 'trackeval_compatibility.patch').write_text(subprocess.check_output(['git','-C',str(EVAL_REPO),'diff'],text=True))
sys.path.insert(0,str(EVAL_REPO))
import trackeval
assert ALIGNMENT_CHECKED
cfg = trackeval.Evaluator.get_default_eval_config()
cfg.update(USE_PARALLEL=False, PRINT_RESULTS=True, PRINT_ONLY_COMBINED=False,
           DISPLAY_LESS_PROGRESS=True, TIME_PROGRESS=False, PLOT_CURVES=False, PRINT_CONFIG=False)
data_cfg = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
data_cfg.update(GT_FOLDER=str(WORK/'gt'), TRACKERS_FOLDER=str(WORK/'trackers'),
                OUTPUT_FOLDER=str(WORK/'metrics'), TRACKERS_TO_EVAL=['official_bytetrack'],
                BENCHMARK='MOT17', SPLIT_TO_EVAL='train', SKIP_SPLIT_FOL=True,
                SEQ_INFO={s:i['validation_frames'] for s,i in info_by_seq.items()},
                DO_PREPROC=True, PRINT_CONFIG=False)
dataset = trackeval.datasets.MotChallenge2DBox(data_cfg)
results, messages = trackeval.Evaluator(cfg).evaluate([dataset],
    [trackeval.metrics.HOTA(), trackeval.metrics.CLEAR(), trackeval.metrics.Identity()])
rows = []
for seq in SEQUENCES + ['COMBINED_SEQ']:
    r = results[dataset.get_name()]['official_bytetrack'][seq]['pedestrian']
    rows.append(dict(sequence=seq, MOTA=100*float(r['CLEAR']['MOTA']),
        IDF1=100*float(r['Identity']['IDF1']), HOTA=100*float(np.mean(r['HOTA']['HOTA'])),
        IDSW=int(r['CLEAR']['IDSW']), FP=int(r['CLEAR']['CLR_FP']), FN=int(r['CLEAR']['CLR_FN'])))
summary = pd.DataFrame(rows)
summary.to_csv(WORK/'summary.csv',index=False)
display(summary.round(2))


## 10. Download the evaluation evidence
The ZIP includes metrics, predictions, split ground truth, settings, patches, checkpoint provenance and dependency versions. It excludes the 5.5 GB archive, JPEG frames and model weights. It is an experiment record, not a submission-ready report. Keep source dataset terms in mind before redistributing annotations publicly.


In [ ]:
metadata = dict(byte_revision=REVISION, trackeval_revision=EVAL_REVISION,
                checkpoint_sha256=weight_hash, python=sys.version,
                torch=torch.__version__, torchvision=torchvision.__version__,
                gpu=torch.cuda.get_device_name(0), sequences=SEQUENCES,
                protocol='Official source modules with explicit half-validation inference wrapper and TrackEval',
                split_rule='original one-based frames N//2+2 through N, reset tracker at validation start',
                deviations=['No original COCO AP computation', 'TrackEval metrics instead of original motmetrics summary',
                            'Advance tracker on empty-detection frames', 'Recorded modern NumPy compatibility edits'],
                limitations=('Seven-scene half-validation reproduction attempt; no leaderboard test evaluation.' if len(SEQUENCES) == 7 else 'One-scene pipeline check; not full MOT17 validation or leaderboard test evaluation.'))
(WORK/'environment.json').write_text(json.dumps(metadata,indent=2))
(WORK/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
archive_path = Path('/content/ByteTrack_official_MOT17_halfval_results.zip')
with zipfile.ZipFile(archive_path,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in WORK.rglob('*'):
        if path.is_file() and 'images' not in path.relative_to(WORK).parts:
            archive.write(path,path.relative_to(WORK))
from google.colab import files
files.download(str(archive_path))
print('Save this ZIP and inspect summary.csv. One sequence first; all seven only after checking the pipeline.')


## What to do afterward

For now, send the downloaded ZIP for review. Keep the model and thresholds unchanged. If the pipeline checks out, use ALL_SEQUENCES=True in a fresh run to obtain an aggregate across seven unique scenes. Keep the original per-scene results and investigate discrepancies before drawing conclusions.

The next scientific comparison is full ByteTrack versus a high-confidence-only ablation on the same cached YOLOX detections. That comparison needs its own carefully controlled notebook; it is not performed here. Do not use the earlier YOLO11 notebooks to claim a YOLOX reproduction.
